# Exercise 1 — Bootstrap Hypothesis Testing on Neural Spike Trains

**Neural Data Science (WP7)** · LMU Biology

---

### Overview

The **bootstrap** is one of the most broadly applicable tools in quantitative
neuroscience: it lets you attach uncertainty bounds to almost any statistic,
even when you cannot derive an analytic null distribution.

In this exercise you will:
1. Load V1 spike-train data and explore its structure
2. Compute a test statistic (mean inter-spike interval)
3. Build a null distribution by shuffling temporal structure
4. Test whether the observed statistic is significantly different from chance

### Learning objectives

- Understand the logic of resampling-based hypothesis testing
- Implement a bootstrap procedure from scratch
- Interpret p-values derived from empirical null distributions
- Visualize and communicate bootstrap results

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

### Load the data

We use single-unit recordings from primary visual cortex (V1), provided by the
CRCNS PVC-8 dataset.  The data is a 4-D binary tensor:

| Axis | Meaning |
|------|--------|
| 0    | Neurons (cells) |
| 1    | Images (stimuli) |
| 2    | Trials per image |
| 3    | Time bins (1 ms each, 0 = stimulus onset) |

Each entry is 0 (no spike) or 1 (spike) in that 1-ms bin.

In [ ]:
data_path = '../../../../sourcedata/data/crcns_pvc8/2.mat'
# Alternative absolute path on the LMU server:
# data_path = '/storage2/arash/teaching/wp7-course/sourcedata/data/crcns_pvc8/2.mat'

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Data not found at {data_path!r}. Adjust data_path to point to crcns_pvc8/2.mat"
    )

mat = scipy.io.loadmat(data_path)
R = mat['resp_train']
n_cells, n_images, n_trials, n_tp = R.shape
print(f'Data shape: {R.shape}')
print(f'  {n_cells} neurons, {n_images} images, {n_trials} trials, {n_tp} time bins (ms)')

---
## Section 1 — Exploring the Data

Before diving into statistics, visualize the raw data to build intuition.
A **spike raster** shows individual trials as rows and time on the x-axis,
with marks where spikes occur.

> **Think about it:** If spikes were placed randomly in time (uniform across
> the 106 ms window), what would the raster look like?  How would it differ
> from structured firing?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Visualize the spike raster
# ──────────────────────────────────────────────────────
# 1. Choose a neuron index (e.g. neuron_idx = 10) and an image index
# 2. Use plt.imshow() to display R[neuron_idx, image_idx, :, :]
#    as a binary image (try cmap='binary')
# 3. Label your axes (trials vs time) and add a title
#
# Then: select your neuron's data across ALL images and trials
#       and reshape to 2-D: trials of shape (n_images * n_trials, n_tp)
# ──────────────────────────────────────────────────────

neuron_idx = ...   # TODO: pick a neuron (e.g. 10)

raise NotImplementedError("Visualize the raster and create the `trials` array")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

`R[neuron_idx, image_idx]` gives you a 2-D array of shape `(n_trials, n_tp)`.
Each row is one trial.  `plt.imshow()` can display this directly as an image
where 1s (spikes) show up in one color and 0s in another.

To get **all** trials for one neuron across all images, index with
`R[neuron_idx]` (shape `(n_images, n_trials, n_tp)`) and then reshape.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
plt.imshow(data_2d, cmap='binary', aspect='auto', interpolation='nearest')
```

To reshape a 3-D array `(n_images, n_trials, n_tp)` into 2-D `(n_images*n_trials, n_tp)`:
```python
trials = R[neuron_idx].reshape(-1, n_tp)
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
neuron_idx = 10
image_idx = 10

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(R[neuron_idx, image_idx, :, :], cmap='binary', aspect='auto',
          interpolation='nearest')
ax.set_ylabel('Trial')
ax.set_xlabel('Time (ms)')
ax.set_title(f'Spike raster — Neuron {neuron_idx}, Image {image_idx}')
plt.tight_layout()
plt.show()

trials = R[neuron_idx].reshape(-1, n_tp)
```
</details>

In [ ]:
# Checkpoint 1
assert trials.shape[1] == n_tp, f"Expected {n_tp} columns, got {trials.shape[1]}"
assert trials.shape[0] == n_images * n_trials, (
    f"Expected {n_images * n_trials} rows, got {trials.shape[0]}"
)
print(f"Checkpoint 1 passed — trials shape: {trials.shape}")

---
## Section 2 — Computing the Test Statistic: Inter-Spike Intervals

An **inter-spike interval (ISI)** is the time between consecutive spikes within
a single trial.  The mean ISI is a simple summary of a neuron's temporal firing
pattern.  If spikes are random, the mean ISI depends only on firing rate.
If there is temporal structure (bursting, oscillations, refractory periods),
the mean ISI will differ from the random expectation.

> **Think about it:** Why is the *mean* ISI a reasonable test statistic here?
> What other summary statistics could you use instead?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Complete the function below
# ──────────────────────────────────────────────────────
# For each trial (row):
#   1. Find the spike times using np.where(row == 1)
#   2. Compute the intervals between consecutive spikes using np.diff()
#   3. Collect all intervals
# Return them concatenated as one array.
# ──────────────────────────────────────────────────────

def compute_isis(trials_2d):
    """Extract all ISIs from a 2-D array of binary spike trains.

    Parameters
    ----------
    trials_2d : ndarray, shape (n_trials_total, n_tp)
        Binary spike trains (0 or 1 per time bin).

    Returns
    -------
    isis : ndarray
        Concatenated inter-spike intervals from all trials.
    """
    all_isis = []
    for i in range(trials_2d.shape[0]):
        # TODO: extract spike times for this trial
        # TODO: if there are at least 2 spikes, compute ISIs with np.diff
        # TODO: append the ISIs to all_isis
        raise NotImplementedError("Complete the ISI computation loop")

    return np.concatenate(all_isis) if all_isis else np.array([])

In [ ]:
# Compute observed ISIs and mean
isis = compute_isis(trials)
observed_mean_isi = np.mean(isis)
print(f'Total ISIs extracted: {len(isis):,}')
print(f'Observed mean ISI:    {observed_mean_isi:.2f} ms')

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

`np.where(array == 1)[0]` returns the **indices** where the condition is True —
i.e., the time bins where spikes occurred.  `np.diff(spike_times)` gives the
differences between consecutive spike times, which are the ISIs.

Some trials may have 0 or 1 spikes — those produce no ISIs and should be skipped.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
spike_times = np.where(trials_2d[i] == 1)[0]   # indices of 1s
if len(spike_times) > 1:
    isis_this_trial = np.diff(spike_times)       # consecutive differences
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
for i in range(trials_2d.shape[0]):
    spike_times = np.where(trials_2d[i] == 1)[0]
    if len(spike_times) > 1:
        all_isis.append(np.diff(spike_times))
```
</details>

In [ ]:
# Checkpoint 2
assert len(isis) > 100, f"Expected many ISIs, got only {len(isis)}"
assert 1 < observed_mean_isi < 100, (
    f"Mean ISI = {observed_mean_isi:.2f} ms seems implausible"
)
print(f"Checkpoint 2 passed — {len(isis):,} ISIs, mean = {observed_mean_isi:.2f} ms")

Plot the ISI distribution to see what we're working with:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(isis, bins=np.arange(0, min(int(isis.max()), 60) + 2, 1), density=True,
        alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(observed_mean_isi, color='red', linestyle='--', linewidth=2,
           label=f'Mean ISI = {observed_mean_isi:.2f} ms')
ax.set_xlabel('Inter-Spike Interval (ms)')
ax.set_ylabel('Probability density')
ax.set_title(f'ISI Distribution — Neuron {neuron_idx}')
ax.set_xlim(0, 60)
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 3 — Building the Null Distribution (Bootstrap)

Now comes the core of the bootstrap:  we need to know what mean ISI values
we would observe **if temporal structure were absent**.  We do this by
*shuffling* each trial's spike train — randomly permuting the time bins.
This preserves the spike count per trial but destroys any temporal ordering.

Repeating this many times (e.g., 1000) gives us an empirical **null
distribution** of the mean ISI.

> **Think about it:** Why does permuting time bins within a trial constitute
> a valid null model for "no temporal structure"?  What aspects of the data
> are preserved, and what is destroyed?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Build the null distribution
# ──────────────────────────────────────────────────────
# For each of n_bootstrap iterations:
#   1. Shuffle each trial's time bins independently
#      (use np.random.permutation on each row)
#   2. Compute the mean ISI from the shuffled data
#      (call your compute_isis function)
#   3. Store the result in null_means[i]
#
# Note: tqdm is not available in the wp7 env.
#       Use a print statement every 250 iterations to
#       track progress.
# ──────────────────────────────────────────────────────

n_bootstrap = 1000
null_means = np.zeros(n_bootstrap)
rng = np.random.default_rng(42)

for i in range(n_bootstrap):
    # TODO: create shuffled version of trials
    # TODO: compute ISIs from shuffled data
    # TODO: store mean ISI in null_means[i]
    raise NotImplementedError("Complete the bootstrap loop")

print('Done!')

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

The shuffle must happen **independently for each trial** (each row of the 2-D
array).  `np.random.permutation(row)` returns a shuffled copy of a 1-D array.
You can build the full shuffled array row by row in a list comprehension.

After shuffling, pass the result through `compute_isis()` exactly as you did
with the original data.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
shuffled = np.array([rng.permutation(row) for row in trials])
isis_null = compute_isis(shuffled)
null_means[i] = np.mean(isis_null)
```

`rng = np.random.default_rng(42)` creates a seeded random generator for
reproducibility.
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
for i in range(n_bootstrap):
    if i % 250 == 0:
        print(f'  iteration {i}/{n_bootstrap} ...')
    shuffled = np.array([rng.permutation(row) for row in trials])
    isis_null = compute_isis(shuffled)
    null_means[i] = np.mean(isis_null) if len(isis_null) > 0 else np.nan
```
</details>

In [ ]:
# Checkpoint 3
assert null_means.shape == (n_bootstrap,), f"Expected shape ({n_bootstrap},)"
assert not np.all(null_means == 0), "null_means is all zeros — bootstrap did not run"
assert np.nanstd(null_means) > 0, "No variance in null — something is wrong"
print(f"Checkpoint 3 passed — null_means range: "
      f"[{np.nanmin(null_means):.2f}, {np.nanmax(null_means):.2f}] ms")

---
## Section 4 — Drawing Conclusions

Now compare your observed test statistic to the null distribution.
The **p-value** is the probability of seeing a value at least as extreme
as the observed one, under the null hypothesis.

> **Think about it:** Should you use a one-sided or two-sided test here?
> What does it mean if the observed mean ISI is *smaller* than the null?
> What about *larger*?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Compute the p-value and visualize
# ──────────────────────────────────────────────────────
# 1. Compute the 5th and 95th percentiles of null_means
#    (use np.nanpercentile)
# 2. Plot the null distribution as a histogram
# 3. Mark the percentile thresholds and the observed value
# 4. Compute a two-sided p-value:
#    fraction of null values at least as far from the
#    null mean as the observed value is
# ──────────────────────────────────────────────────────

raise NotImplementedError("Compute p-value and create the comparison plot")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

A **two-sided p-value** asks: "How often does the null produce a value
at least as far from its center as our observation?"

Compute the distance of each null sample from the null mean,
and compare to the distance of the observed value from the null mean.
The p-value is the fraction where `|null - null_mean| >= |observed - null_mean|`.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
lower_5 = np.nanpercentile(null_means, 5)
upper_95 = np.nanpercentile(null_means, 95)
null_center = np.nanmean(null_means)
p_value = np.nanmean(
    np.abs(null_means - null_center) >= np.abs(observed_mean_isi - null_center)
)
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
lower_5 = np.nanpercentile(null_means, 5)
upper_95 = np.nanpercentile(null_means, 95)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_means[~np.isnan(null_means)], bins=30, density=True,
        alpha=0.7, color='steelblue', edgecolor='black', label='Null distribution')
ax.axvline(lower_5, color='orange', linestyle='--', label=f'5th pctl = {lower_5:.2f}')
ax.axvline(upper_95, color='orange', linestyle='--', label=f'95th pctl = {upper_95:.2f}')
ax.axvline(observed_mean_isi, color='red', linewidth=2.5,
           label=f'Observed = {observed_mean_isi:.2f}')
ax.set_xlabel('Mean ISI (ms)')
ax.set_ylabel('Probability density')
ax.set_title('Bootstrap Null Distribution vs Observed')
ax.legend()
plt.tight_layout()
plt.show()

null_center = np.nanmean(null_means)
p_value = np.nanmean(
    np.abs(null_means - null_center) >= np.abs(observed_mean_isi - null_center)
)
print(f'Two-sided p-value: {p_value:.4f}')
```
</details>

In [ ]:
# Checkpoint 4
assert 0 <= p_value <= 1, f"p-value = {p_value} is outside [0, 1]"
print(f"Checkpoint 4 passed — p-value = {p_value:.4f}")
if p_value < 0.05:
    print("  Result: Significant at alpha = 0.05 — temporal structure is non-random!")
else:
    print("  Result: Not significant at alpha = 0.05.")

---
## Reflection

Congratulations — you have completed a full bootstrap hypothesis test!
Take a moment to think about these questions (no code required):

1. **Robustness:** You tested one neuron.  Would you expect the result to hold
   for all neurons in the dataset?  How would you extend the analysis to
   check this systematically?

2. **Test statistic choice:** We used the *mean* ISI.  What if you used the
   *median* ISI, or the *coefficient of variation* of ISIs?  Would the
   conclusion change?  Why might one statistic be more sensitive than another?

3. **Assumptions:** The null hypothesis preserves the spike *count* per trial
   but randomizes timing.  What alternative null models could you construct?
   For example, what if you shuffled spikes *across* trials instead of
   *within* trials — what would that test?